### SETUP

In [1]:
from torch.cuda import empty_cache
from ultralytics import YOLO
# For 3.13 versions or lower
try:
    import albumentations as A

    custom_transforms = [
        # Blur variations
        A.OneOf(
            [
                A.MotionBlur(blur_limit=7, p=1.0),
                A.MedianBlur(blur_limit=7, p=1.0),
                A.GaussianBlur(blur_limit=7, p=1.0),
            ],
            p=0.3,
        ),

        # Noise variations
        A.OneOf(
            [
                A.GaussNoise(var_limit=(10.0, 50.0), p=1.0),
                A.ISONoise(color_shift=(0.01, 0.05), intensity=(0.1, 0.5), p=1.0),
            ],
            p=0.2,
        ),

        # Color and contrast adjustments
        A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8), p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.5),
        A.HueSaturationValue(hue_shift_limit=20, sat_shift_limit=30, val_shift_limit=20, p=0.5),

        # Simulate occlusions
        A.CoarseDropout(
            max_holes=8, max_height=32, max_width=32, min_holes=1, min_height=8, min_width=8, fill_value=0, p=0.2
        ),
    ]
except ModuleNotFoundError:
    print('Albumentations only supports Python versions 3.9 - 3.14.0')
    custom_transforms = []

C:\Users\Bui Thien Nghia\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Bui Thien Nghia\AppData\Local\Temp\ipykernel_11160\2616644446.py:21: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(10.0, 50.0), p=1.0),
C:\Users\Bui Thien Nghia\AppData\Local\Temp\ipykernel_11160\2616644446.py:33: UserWarning: Argument(s) 'max_holes, max_height, max_width, min_holes, min_height, min_width, fill_value' are not valid for transform CoarseDropout
  A.CoarseDropout(


### TRAINING

In [ ]:
model = YOLO('yolo26s.pt')
model.train(
    data='data.yaml',
    cfg='trainconfig.yaml',
    augmentations=custom_transforms
)

### TUNING

In [5]:
model = YOLO('model/obj_tuned_416/weights/best.pt')
model.train(
    data='data-tune.yaml',
    cfg='trainconfig.yaml',
    augmentations=custom_transforms,
)

Ultralytics 8.4.13  Python-3.12.0 torch-2.10.0+cu130 CUDA:0 (NVIDIA GeForce RTX 5060 Laptop GPU, 8151MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, augmentations=[OneOf([
  MotionBlur(p=1.0, allow_shifted=True, angle_range=(0.0, 360.0), blur_limit=(3, 7), direction_range=(-1.0, 1.0)),
  MedianBlur(p=1.0, blur_limit=(3, 7)),
  GaussianBlur(p=1.0, blur_limit=(0, 7), sigma_limit=(0.5, 3.0)),
], p=0.3), OneOf([
  GaussNoise(p=1.0, mean_range=(0.0, 0.0), noise_scale_factor=1.0, per_channel=True, std_range=(0.2, 0.44)),
  ISONoise(p=1.0, color_shift=(0.01, 0.05), intensity=(0.1, 0.5)),
], p=0.2), CLAHE(p=0.5, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8)), RandomBrightnessContrast(p=0.5, brightness_by_max=True, brightness_limit=(-0.3, 0.3), contrast_limit=(-0.3, 0.3), ensure_safe_range=False), HueSaturationValue(p=0.5, hue_shift_limit=(-20.0, 20.0), sat_shift_limit=(-30.0, 30.0), val_shift_limit=(-20.0, 20.0)), CoarseDropout(p=0.2, fill=0.0, fill_mask=None, hole_

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0,  2,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x00000283C6FE9F10>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,   

### VALIDATING

In [ ]:
model = YOLO('model/full_tuned/weights/best.pt')
model.val(
    data='data-tune.yaml',
    cfg='trainconfig.yaml'
)

### PREDICTING

In [ ]:
from ultralytics import YOLO
from glob import glob

paths = glob(r'D:\BFMC2026\BFMC-Traffic-Signal-1\train\images\*.jpg') + glob(r'D:\BFMC2026\BFMC-Traffic-Signal-1\train\images\*.png')
print(len(paths))
model = YOLO('model/full_new_trainer/weights/best.pt')

for i in range(len(paths)):
    result = model.predict(paths[i])
    for r in result:
        print(r.boxes)
    result[0].save()

### EXPORT MODEL

In [3]:
model = YOLO('model/obj_tuned/weights/best.pt')
# FP16 ONNX
model.export(
    format='onnx',
    imgsz=320,
    dynamic=False,
    half=True,
    simplify=True
)

# FP16 TENSORRT
model.export(
    format='engine',
    imgsz=320,
    half=True,
    data='data-tune.yaml',
)

# INT8 TENSORRT (RUN THIS ON JETSON ORIN)
# model.export(
#     format='engine',
#     imgsz=320,
#     half=True,
#     int8=True,
#     data='data-tune.yaml',
# )

Ultralytics 8.4.13  Python-3.12.0 torch-2.10.0+cu130 CPU (Intel Core(TM) i5-10210U 1.60GHz)
YOLO26s summary (fused): 122 layers, 9,471,759 parameters, 0 gradients, 20.6 GFLOPs

PyTorch: starting from 'model\obj_tuned\weights\best.pt' with input shape (1, 3, 320, 320) BCHW and output shape(s) (1, 300, 6) (19.4 MB)

ONNX: starting export with onnx 1.20.1 opset 22...
ONNX: slimming with onnxslim 0.1.84...
ONNX: converting to FP16...
ONNX: export success  2.2s, saved as 'model\obj_tuned\weights\best.onnx' (18.2 MB)

Export complete (2.5s)
Results saved to C:\Users\Bui Thien Nghia\Documents\PERSONAL FILES 2\Cholibi-BFMC2026\lab_n_archives\model\obj_tuned\weights
Predict:         yolo predict task=detect model=model\obj_tuned\weights\best.onnx imgsz=320 half
Validate:        yolo val task=detect model=model\obj_tuned\weights\best.onnx imgsz=320 data=data-tune.yaml half 
Visualize:       https://netron.app
WARNING TensorRT requires GPU export, automatically assigning device=0
Ultralytics 8.4.

c:\Users\Bui Thien Nghia\Documents\PERSONAL FILES 2\Cholibi-BFMC2026\lab_n_archives\venv\Lib\site-packages\torch\onnx\_internal\torchscript_exporter\symbolic_opset11.py:954: UserWarning: Exporting aten::index operator of advanced indexing in opset 20 is achieved by combination of multiple ONNX operators, including Reshape, Transpose, Concat, and Gather. If indices include negative values, the exported graph will produce incorrect results.
  return opset9.index(g, self, index)


ONNX: slimming with onnxslim 0.1.84...
ONNX: export success  2.3s, saved as 'model\obj_tuned\weights\best.onnx' (36.3 MB)

TensorRT: starting export with TensorRT 10.15.1.29...
TensorRT: input "images" with shape(1, 3, 320, 320) DataType.FLOAT
TensorRT: output "output0" with shape(1, 300, 6) DataType.FLOAT
TensorRT: building FP16 engine as model\obj_tuned\weights\best.engine
TensorRT: export success  60.7s, saved as 'model\obj_tuned\weights\best.engine' (19.7 MB)

Export complete (62.0s)
Results saved to C:\Users\Bui Thien Nghia\Documents\PERSONAL FILES 2\Cholibi-BFMC2026\lab_n_archives\model\obj_tuned\weights
Predict:         yolo predict task=detect model=model\obj_tuned\weights\best.engine imgsz=320 half
Validate:        yolo val task=detect model=model\obj_tuned\weights\best.engine imgsz=320 data=data-tune.yaml half 
Visualize:       https://netron.app


'model\\obj_tuned\\weights\\best.engine'

### DEBUGGING

In [62]:
from systemMode import SystemModeRebuilt

class StateChanger:
    def __init__(self):
        self.cur_state = SystemModeRebuilt.LANE_KEEPING_NORMAL
        self.classes = [
            'pedestrian',
            'cyclist',
            'car',
            'bus',
            'truck',
            'red_light',
            'yellow_light',
            'green_light',
            'crosswalk_sign',
            'enter_highway_sign',
            'leave_highway_sign',
            'oneway_sign',
            'parking_sign',
            'priority_sign',
            'noentry_sign',
            'roundabout_sign',
            'stop_sign'
        ]
        self.idx_to_cls = {
            0: 'pedestrian',
            1: 'cyclist',
            2: 'car',
            3: 'bus',
            4: 'truck',
            5: 'red_light',
            6: 'yellow_light',
            7: 'green_light',
            8: 'crosswalk_sign',
            9: 'enter_highway_sign',
            10: 'leave_highway_sign',
            11: 'oneway_sign',
            12: 'parking_sign',
            13: 'priority_sign',
            14: 'noentry_sign',
            15: 'roundabout_sign',
            16: 'stop_sign'
        }
        # To be tuned
        self.det_threshold = {
            'pedestrian': 7,
            'cyclist': 7,
            'car': 7,
            'bus': 7,
            'truck': 7,
            'red_light': 7,
            'yellow_light': 7,
            'green_light': 7,
            'crosswalk_sign': 7,
            'enter_highway_sign': 7,
            'leave_highway_sign': 7,
            'oneway_sign': 7,
            'parking_sign': 7,
            'priority_sign': 7,
            'noentry_sign': 7,
            'roundabout_sign': 7,
            'stop_sign': 7
        }
        self.cur_dets = {key: 0 for key in [
            'pedestrian',
            'cyclist',
            'car',
            'bus',
            'truck',
            'red_light',
            'yellow_light',
            'green_light',
            'crosswalk_sign',
            'enter_highway_sign',
            'leave_highway_sign',
            'oneway_sign',
            'parking_sign',
            'priority_sign',
            'noentry_sign',
            'roundabout_sign',
            'stop_sign'
        ]}

    def record_detection(self, idxes, boxes):
        def get_max_cnt(cls, coeff=1):
            return int(self.det_threshold[cls] * coeff)
        
        # To be tuned
        def aspect_ratio_met(box, aspect_ratio, err_rate):
            return aspect_ratio * (1 - err_rate) <= box[-2] / box[-1] <= aspect_ratio * (1 + err_rate)

        dets = [self.idx_to_cls[i] for i in idxes]
        accepted_dets = []
        for d, b in zip(dets, boxes):
            if d in self.classes[:5]:
                accepted_dets.append(d)
            elif d in self.classes[5:7] and aspect_ratio_met(b, 1/3.5, 0.4):
                accepted_dets.append(d)
            elif d in self.classes[7:]:
                if d in ['enter_highway_sign', 'leave_highway_sign'] and aspect_ratio_met(b, 2/3, 0.4):
                    accepted_dets.append(d)
                elif aspect_ratio_met(b, 1/1, 0.4):
                    accepted_dets.append(d)
        
        for c in list(self.cur_dets.keys()):
            if c in accepted_dets:
                self.cur_dets[c] += 1 if self.cur_dets[c] < get_max_cnt(c, 2) else 0
            else:
                self.cur_dets[c] -= 1 if self.cur_dets[c] > 0 else 0

    def change_state(self):
        # threshold check util
        def threshold_met(cls):
            return self.cur_dets[cls] >= self.det_threshold[cls]
        
        # traffic light handling
        if threshold_met('red_light'):
            self.cur_state = SystemModeRebuilt.STOP
        elif threshold_met('yellow_light') and self.cur_state != SystemModeRebuilt.STOP:
            self.cur_state = SystemModeRebuilt.LANE_KEEPING_SLOW
        elif threshold_met('green_light'):
            self.cur_state = SystemModeRebuilt.LANE_KEEPING_NORMAL

        # traffic sign handling
        elif threshold_met('stop_sign'):
            self.cur_state = SystemModeRebuilt.STOP
        elif threshold_met('noentry_sign'): # add intersection detection here
            self.cur_state = SystemModeRebuilt.STOP
        elif threshold_met('crosswalk_sign'):
            self.cur_state = SystemModeRebuilt.LANE_KEEPING_SLOW
        elif threshold_met('oneway_sign'): # consider adding a strict go forward only OR construct for this case
            self.cur_state = SystemModeRebuilt.LANE_KEEPING_NORMAL
        elif threshold_met('priority_sign'):
            self.cur_state = SystemModeRebuilt.LANE_KEEPING_NORMAL
        elif threshold_met('leave_highway_sign'):
            self.cur_state = SystemModeRebuilt.LANE_KEEPING_NORMAL
        elif threshold_met('roundabout_sign'): # build a turn recognition based on route detection
            self.cur_state = SystemModeRebuilt.LANE_KEEPING_NORMAL or SystemModeRebuilt.TURN
        elif threshold_met('enter_highway_sign'):
            self.cur_state = SystemModeRebuilt.LANE_KEEPING_FAST
        elif threshold_met('parking_sign'):
            self.cur_state = SystemModeRebuilt.PARKING

        # object handling
        elif threshold_met('pedestrian') or threshold_met('cyclist'):
            self.cur_state = SystemModeRebuilt.STOP
        elif threshold_met('car') or threshold_met('truck') or threshold_met('bus'): # build a decisor based on movement/distance tracking
            self.cur_state = SystemModeRebuilt.OVERTAKING or SystemModeRebuilt.TAILING

        # No detection
        else:
            self.cur_state = SystemModeRebuilt.LANE_KEEPING_NORMAL

    def _get_state(self):
        return self.cur_state

In [49]:
model = YOLO('model/obj_tuned/weights/best.pt')
video_path = r'C:\Users\Bui Thien Nghia\Documents\PERSONAL FILES 2\Cholibi-BFMC2026\lab_n_archives\Records\bfmc2020_online_3.avi'

results = model.track(
    source=video_path,
    # show=True,
    half=True,
    imgsz=416,
    conf=0.75,
    vid_stride=1,
    # save=True,
    verbose=False
)

WARNING 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs



In [63]:
import cv2
import time

max = 0
res = None
coordinates = (10, 50) # Bottom-left corner
font = cv2.FONT_HERSHEY_SIMPLEX
fontScale = 1.5
outline_color = (255, 255, 255)
color = (0, 0, 0) # Green color in BGR
thickness = 2

mode_changer = StateChanger()
for i, r in enumerate(results):
    mode_changer.record_detection(r.boxes.cls.tolist(), r.boxes.xywhn.tolist())
    mode_changer.change_state()
    cur_state = mode_changer._get_state()

    img = r.plot()
    cv2.putText(img, cur_state.value['mode'], coordinates, font, fontScale, outline_color, thickness + 4, cv2.LINE_AA)
    cv2.putText(img, cur_state.value['mode'], coordinates, font, fontScale, color, thickness, cv2.LINE_AA)
    cv2.imshow('bruh', img)
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

    if len(r.boxes.cls) > max:
        res = r
        max = len(r.boxes.cls)
    time.sleep(0.03)

cv2.destroyAllWindows()

In [52]:
# print(res)
print(res.boxes)
# print(res.names)

ultralytics.engine.results.Boxes object with attributes:

cls: tensor([12., 12.])
conf: tensor([0.8872, 0.8130])
data: tensor([[1.2614e+03, 4.1788e+02, 1.3847e+03, 5.2854e+02, 2.9000e+01, 8.8721e-01, 1.2000e+01],
        [1.0016e+03, 3.8867e+02, 1.0465e+03, 4.3272e+02, 3.1000e+01, 8.1299e-01, 1.2000e+01]])
id: tensor([29., 31.])
is_track: True
orig_shape: (1232, 1640)
shape: torch.Size([2, 7])
xywh: tensor([[1323.0610,  473.2097,  123.3423,  110.6673],
        [1024.0544,  410.6914,   44.9862,   44.0521]])
xywhn: tensor([[0.8067, 0.3841, 0.0752, 0.0898],
        [0.6244, 0.3334, 0.0274, 0.0358]])
xyxy: tensor([[1261.3899,  417.8761, 1384.7322,  528.5434],
        [1001.5614,  388.6653, 1046.5476,  432.7175]])
xyxyn: tensor([[0.7691, 0.3392, 0.8443, 0.4290],
        [0.6107, 0.3155, 0.6381, 0.3512]])
